In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

# Load data
train_path = "/kaggle/input/lmsys-chatbot-arena/train.csv"
test_path = "/kaggle/input/lmsys-chatbot-arena/test.csv"
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)




In [2]:
# Build a simple text feature: concatenate prompt and both responses
def concatenate_text(row):
    return f"{row['prompt']} {row['response_a']} {row['response_b']}"


train_text = train_df.apply(concatenate_text, axis=1).tolist()
test_text = test_df.apply(concatenate_text, axis=1).tolist()

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_all = vectorizer.fit_transform(train_text)  # fit on train only
X_test = vectorizer.transform(test_text)



In [3]:
# Split train for quick validation
y = train_df[["winner_model_a", "winner_model_b", "winner_tie"]].values
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y.argmax(axis=1)
)

# Train linear model (kept unchanged)
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Predict on validation and convert to probabilities
val_pred_raw = lin_reg.predict(X_val)


def softmax(z):
    z_exp = np.exp(z - np.max(z, axis=1, keepdims=True))
    return z_exp / np.sum(z_exp, axis=1, keepdims=True)


val_pred_prob = softmax(val_pred_raw)

# Compute log‑loss (lower is better)
val_score = log_loss(y_val, val_pred_prob, eps=1e-15)
print(f"Validation LogLoss: {val_score}")



Validation LogLoss: 1.1017964926612118


In [4]:
# Retrain on the full training set
lin_reg_full = LinearRegression()
lin_reg_full.fit(X_all, y)

# Predict on test set
test_pred_raw = lin_reg_full.predict(X_test)
test_pred_prob = softmax(test_pred_raw)



In [5]:
# Build submission with correct column names
submission = pd.DataFrame(
    {
        "id": test_df["id"],
        "winner_model_a": test_pred_prob[:, 0],
        "winner_model_b": test_pred_prob[:, 1],
        "winner_tie": test_pred_prob[:, 2],
    }
)



In [6]:
# Write the submission file
submission.to_csv("submission.csv", index=False)